# Week 12, Day 5: Learning rules as optimisation: perceptron, delta rule and tempotron

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w012/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1958)
n_neurons, n_trials = 20, 200
mean_A = rng.uniform(5, 15, n_neurons)
mean_B = mean_A + rng.normal(0, 0.8, n_neurons)                         # the two stimuli differ only slightly
labels = np.where(np.arange(n_trials) % 2 == 0, 1, -1)                # stimulus A (+1) or B (-1)
rates = np.where(labels[:, None] > 0, mean_A, mean_B) + rng.normal(0, 1.0, (n_trials, n_neurons))
Xp = np.column_stack([rates - rates.mean(axis=0), np.ones(n_trials)])   # centred responses plus a bias input
print(Xp.shape)

### Exercise: Train a perceptron decoder

Train a perceptron on the population responses: sweep through the trials in order, repeatedly, updating w ← w + y x on every mistake, until a full pass makes no mistakes (at most 300 passes). Count all the updates in `mistakes` and store the weights in `w_p`.

In [ ]:
w_p = np.zeros(Xp.shape[1])
mistakes = 0
for epoch in range(300):
    errors = 0
    for x, yv in zip(Xp, labels):
        if ...:
            ...
    if errors == 0:
        break
gamma = np.min(labels * (Xp @ w_p)) / np.linalg.norm(w_p)
R = np.max(np.linalg.norm(Xp, axis=1))
print(f"converged after {epoch + 1} passes with {mistakes} mistakes;  bound (R/gamma)^2 = {(R / gamma) ** 2:.0f}")

In [ ]:
import numpy as np
def test_separates():
    "after training, the perceptron classifies every training trial correctly"
    assert np.all(np.sign(Xp @ w_p) == labels)

def test_bound():
    "the number of mistakes respects Novikoff's bound computed with the final weights' margin"
    gamma = np.min(labels * (Xp @ w_p)) / np.linalg.norm(w_p)
    R = np.max(np.linalg.norm(Xp, axis=1))
    assert mistakes <= (R / gamma) ** 2

test_separates()
test_bound()
print("All checks passed")

### Exercise: The delta rule finds least squares

Treat the stimulus labels (±1) as a continuous target and learn linear weights online with the delta rule: 50 passes through the trials in a random order each pass, learning rate 0.001. Compare the result `w_d` with the least-squares solution `w_ls`.

In [ ]:
w_ls = np.linalg.lstsq(Xp, labels.astype(float), rcond=None)[0]
w_d = np.zeros(Xp.shape[1])
for epoch in range(50):
    for i in rng.permutation(n_trials):
        x, yv = Xp[i], labels[i]
        w_d = ...
print("relative distance from least squares:", round(np.linalg.norm(w_d - w_ls) / np.linalg.norm(w_ls), 4),
      "  training accuracy:", np.mean(np.sign(Xp @ w_d) == labels))

In [ ]:
import numpy as np
def test_close():
    "after 50 online passes, the delta-rule weights are within 5% of the least-squares solution"
    assert np.linalg.norm(w_d - w_ls) / np.linalg.norm(w_ls) < 0.05

test_close()
print("All checks passed")

### Exercise: Train a tempotron

Precompute, for each pattern, the matrix `Km[p]` of PSPs (afferents by time) from `K`. Then train: for each pattern in a random order, find the time of peak voltage; if the decision (peak ≥ 1) is wrong, add or subtract 0.02 × the PSPs at that time from the weights. Stop after an epoch with no errors (at most 300 epochs).

In [ ]:
rng_t = np.random.default_rng(2006)
N_aff, P, T_ms = 50, 20, 500
tau, tau_s = 15.0, 3.75
t_peak = tau * tau_s / (tau - tau_s) * np.log(tau / tau_s)
V0 = 1 / (np.exp(-t_peak / tau) - np.exp(-t_peak / tau_s))       # normalise the PSP peak to 1
def K(s):
    return np.where(s >= 0, V0 * (np.exp(-s / tau) - np.exp(-s / tau_s)), 0.0)

times = np.arange(T_ms)                                        # 1 ms resolution
spike_times = rng_t.uniform(0, T_ms, (P, N_aff))               # one spike per afferent per pattern
pattern_labels = np.array([1] * 10 + [0] * 10)
Km = np.array([K(times[None, :] - spike_times[p][:, None]) for p in range(P)])   # (P, N_aff, T_ms)

w_t = rng_t.normal(0, 0.05, N_aff)
for epoch in range(300):
    errors = 0
    for p in rng_t.permutation(P):
        V = w_t @ Km[p]
        t_max = int(np.argmax(V))
        fired = V[t_max] >= 1
        if fired != bool(pattern_labels[p]):
            errors += 1
            ...
    if errors == 0:
        break

fig, ax = plt.subplots(1, 2, figsize=(9, 3), sharey=True)
for a, p in zip(ax, [0, 10]):
    a.plot(times, w_t @ Km[p]); a.axhline(1, ls="--", color="r"); a.set_title("positive pattern" if p == 0 else "negative pattern"); a.set_xlabel("time (ms)")
plt.tight_layout(); plt.show()
print("learned all patterns after", epoch + 1, "epochs")

In [ ]:
import numpy as np
def test_kernel():
    "the PSP kernel is zero before the spike and peaks at 1"
    s = np.arange(-5, 100, 0.01)
    assert np.all(K(s[s < 0]) == 0) and abs(K(s).max() - 1) < 1e-3

def test_learned():
    "after training, the tempotron fires for every positive pattern and none of the negative ones"
    fired = np.array([(w_t @ Km[p]).max() >= 1 for p in range(P)])
    assert np.array_equal(fired, pattern_labels.astype(bool))

test_kernel()
test_learned()
print("All checks passed")